# 智能邮件助手（EmailSmartAssistant）

这个Notebook实现了一个完整的智能邮件处理系统，包括：
- 邮件自动分类
- 智能回复草稿生成
- 重要事项智能提醒
- 邮件关键信息提取
- 邮件归档整理

## 1. 导入必要的库

In [ ]:
import imaplib
import smtplib
import email
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.header import decode_header
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import jieba
from textblob import TextBlob
from langdetect import detect
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import dateparser
import arrow
from jinja2 import Template
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

console = Console()
print("✅ 所有库导入成功！")

## 2. 配置加载

In [ ]:
# 加载配置文件
def load_config():
    try:
        with open('config/email_config.json', 'r', encoding='utf-8') as f:
            config = json.load(f)
        console.print("✅ 配置文件加载成功", style="green")
        return config
    except FileNotFoundError:
        console.print("❌ 配置文件未找到，请检查 config/email_config.json", style="red")
        return None

# 加载回复模板
def load_templates():
    try:
        with open('templates/reply_templates.json', 'r', encoding='utf-8') as f:
            templates = json.load(f)
        console.print("✅ 回复模板加载成功", style="green")
        return templates
    except FileNotFoundError:
        console.print("❌ 模板文件未找到，请检查 templates/reply_templates.json", style="red")
        return None

config = load_config()
templates = load_templates()

## 3. 邮件连接和获取类

In [ ]:
class EmailConnector:
    def __init__(self, email_config):
        self.config = email_config
        self.imap_conn = None
        self.smtp_conn = None
    
    def connect_imap(self):
        """连接IMAP服务器"""
        try:
            self.imap_conn = imaplib.IMAP4_SSL(self.config['imap_server'], self.config['imap_port'])
            self.imap_conn.login(self.config['email'], self.config['password'])
            console.print(f"✅ IMAP连接成功: {self.config['email']}", style="green")
            return True
        except Exception as e:
            console.print(f"❌ IMAP连接失败: {str(e)}", style="red")
            return False
    
    def get_emails(self, folder='INBOX', limit=50):
        """获取邮件列表"""
        if not self.imap_conn:
            if not self.connect_imap():
                return []
        
        try:
            self.imap_conn.select(folder)
            status, messages = self.imap_conn.search(None, 'ALL')
            
            if status != 'OK':
                return []
            
            email_ids = messages[0].split()
            # 获取最新的邮件
            email_ids = email_ids[-limit:] if len(email_ids) > limit else email_ids
            
            emails = []
            for email_id in tqdm(email_ids, desc="获取邮件"):
                status, msg_data = self.imap_conn.fetch(email_id, '(RFC822)')
                if status == 'OK':
                    email_message = email.message_from_bytes(msg_data[0][1])
                    emails.append(self.parse_email(email_message, email_id.decode()))
            
            return emails
        except Exception as e:
            console.print(f"❌ 获取邮件失败: {str(e)}", style="red")
            return []
    
    def parse_email(self, email_message, email_id):
        """解析邮件内容"""
        # 解码邮件头
        def decode_mime_words(s):
            return ''.join(
                word.decode(encoding or 'utf-8') if isinstance(word, bytes) else word
                for word, encoding in decode_header(s)
            )
        
        subject = decode_mime_words(email_message['Subject'] or '')
        sender = decode_mime_words(email_message['From'] or '')
        date = email_message['Date']
        
        # 获取邮件正文
        body = ""
        if email_message.is_multipart():
            for part in email_message.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        body = part.get_payload(decode=True).decode('utf-8')
                        break
                    except:
                        continue
        else:
            try:
                body = email_message.get_payload(decode=True).decode('utf-8')
            except:
                body = str(email_message.get_payload())
        
        return {
            'id': email_id,
            'subject': subject,
            'sender': sender,
            'date': date,
            'body': body,
            'raw_message': email_message
        }
    
    def close_connections(self):
        """关闭连接"""
        if self.imap_conn:
            self.imap_conn.close()
            self.imap_conn.logout()
        if self.smtp_conn:
            self.smtp_conn.quit()

print("✅ 邮件连接器类定义完成")

## 4. 邮件分类器

In [ ]:
class EmailClassifier:
    def __init__(self, config):
        self.config = config
        self.classification_rules = config['classification_rules']
        self.priority_rules = config['priority_rules']
    
    def classify_email_type(self, email_data):
        """分类邮件类型"""
        subject = email_data['subject'].lower()
        body = email_data['body'].lower()
        sender = email_data['sender'].lower()
        
        text_content = f"{subject} {body}"
        
        # 检查垃圾邮件关键词
        spam_score = sum(1 for keyword in self.classification_rules['spam_keywords'] 
                        if keyword in text_content)
        if spam_score >= 2:
            return 'spam'
        
        # 检查工作邮件关键词
        work_score = sum(1 for keyword in self.classification_rules['work_keywords'] 
                        if keyword in text_content)
        
        # 检查客户咨询关键词
        customer_score = sum(1 for keyword in self.classification_rules['customer_keywords'] 
                           if keyword in text_content)
        
        # 检查个人邮件关键词
        personal_score = sum(1 for keyword in self.classification_rules['personal_keywords'] 
                           if keyword in text_content)
        
        # 根据得分确定类型
        scores = {
            'work': work_score,
            'customer': customer_score,
            'personal': personal_score
        }
        
        return max(scores, key=scores.get) if max(scores.values()) > 0 else 'other'
    
    def classify_priority(self, email_data):
        """分类邮件优先级"""
        subject = email_data['subject'].lower()
        body = email_data['body'].lower()
        sender = email_data['sender']
        
        text_content = f"{subject} {body}"
        
        # 检查高优先级发件人
        if any(priority_sender in sender for priority_sender in self.priority_rules['high_priority_senders']):
            return 'high'
        
        # 检查高优先级关键词
        high_priority_score = sum(1 for keyword in self.priority_rules['high_priority_keywords'] 
                                 if keyword in text_content)
        if high_priority_score > 0:
            return 'high'
        
        # 检查低优先级关键词
        low_priority_score = sum(1 for keyword in self.priority_rules['low_priority_keywords'] 
                                if keyword in text_content)
        if low_priority_score > 0:
            return 'low'
        
        return 'medium'
    
    def classify_sender_type(self, email_data):
        """分类发件人类型"""
        sender = email_data['sender'].lower()
        
        # 简单的发件人分类逻辑
        if any(domain in sender for domain in ['@company.com', '@work.com']):
            return 'colleague'
        elif 'noreply' in sender or 'no-reply' in sender:
            return 'system'
        elif any(keyword in sender for keyword in ['service', 'support', 'info']):
            return 'customer_service'
        else:
            return 'external'
    
    def classify_email(self, email_data):
        """完整的邮件分类"""
        return {
            'type': self.classify_email_type(email_data),
            'priority': self.classify_priority(email_data),
            'sender_type': self.classify_sender_type(email_data)
        }

print("✅ 邮件分类器定义完成")

## 5. 关键信息提取器

In [ ]:
class InformationExtractor:
    def __init__(self):
        # 时间相关的正则表达式
        self.date_patterns = [
            r'\d{4}[-/]\d{1,2}[-/]\d{1,2}',  # 2024-01-01 或 2024/01/01
            r'\d{1,2}[-/]\d{1,2}[-/]\d{4}',  # 01-01-2024 或 01/01/2024
            r'\d{1,2}月\d{1,2}日',           # 1月1日
            r'\d{1,2}/\d{1,2}',              # 1/1
        ]
        
        # 时间相关的关键词
        self.time_keywords = [
            '截止', 'deadline', '到期', '完成时间', '交付时间',
            '会议时间', '约定时间', '预定', '安排在'
        ]
        
        # 待办事项关键词
        self.todo_keywords = [
            '需要', '请', '要求', '完成', '处理', '准备',
            'need', 'please', 'require', 'complete', 'prepare'
        ]
    
    def extract_dates(self, text):
        """提取文本中的日期"""
        dates = []
        
        # 使用正则表达式提取日期
        for pattern in self.date_patterns:
            matches = re.findall(pattern, text)
            dates.extend(matches)
        
        # 使用dateparser解析更复杂的日期表达
        sentences = text.split('。')
        for sentence in sentences:
            if any(keyword in sentence for keyword in self.time_keywords):
                parsed_date = dateparser.parse(sentence)
                if parsed_date:
                    dates.append(parsed_date.strftime('%Y-%m-%d'))
        
        return list(set(dates))  # 去重
    
    def extract_todos(self, text):
        """提取待办事项"""
        todos = []
        sentences = text.split('。')
        
        for sentence in sentences:
            if any(keyword in sentence for keyword in self.todo_keywords):
                # 清理句子
                clean_sentence = sentence.strip()
                if len(clean_sentence) > 5:  # 过滤太短的句子
                    todos.append(clean_sentence)
        
        return todos
    
    def extract_contacts(self, text):
        """提取联系人信息"""
        contacts = {
            'emails': [],
            'phones': []
        }
        
        # 提取邮箱地址
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        contacts['emails'] = re.findall(email_pattern, text)
        
        # 提取电话号码
        phone_patterns = [
            r'1[3-9]\d{9}',  # 中国手机号
            r'\d{3}-\d{4}-\d{4}',  # 格式化电话
            r'\(\d{3}\)\s*\d{3}-\d{4}'  # 美式电话格式
        ]
        
        for pattern in phone_patterns:
            contacts['phones'].extend(re.findall(pattern, text))
        
        return contacts
    
    def generate_summary(self, email_data):
        """生成邮件摘要"""
        subject = email_data['subject']
        body = email_data['body']
        sender = email_data['sender']
        
        # 提取关键信息
        dates = self.extract_dates(body)
        todos = self.extract_todos(body)
        contacts = self.extract_contacts(body)
        
        # 生成摘要
        summary = {
            'subject': subject,
            'sender': sender,
            'key_dates': dates,
            'todo_items': todos[:3],  # 最多3个待办事项
            'contacts': contacts,
            'body_preview': body[:200] + '...' if len(body) > 200 else body
        }
        
        return summary

print("✅ 信息提取器定义完成")

## 6. 智能回复生成器

In [ ]:
class ReplyGenerator:
    def __init__(self, templates, config):
        self.templates = templates
        self.config = config
        self.reply_settings = config['reply_settings']
    
    def detect_language(self, text):
        """检测文本语言"""
        try:
            lang = detect(text)
            return 'zh' if lang == 'zh-cn' else 'en'
        except:
            return 'zh'  # 默认中文
    
    def select_template(self, email_classification, email_data):
        """根据邮件分类选择合适的模板"""
        email_type = email_classification['type']
        
        # 根据邮件类型选择模板
        if email_type == 'work':
            if '会议' in email_data['subject'] or 'meeting' in email_data['subject'].lower():
                return 'work_meeting'
            else:
                return 'general_acknowledgment'
        elif email_type == 'customer':
            return 'customer_inquiry'
        else:
            return 'general_acknowledgment'
    
    def generate_reply(self, email_data, email_classification):
        """生成回复草稿"""
        # 选择模板
        template_key = self.select_template(email_classification, email_data)
        
        # 检测语言
        language = self.detect_language(email_data['body'])
        
        # 确定语气（正式/非正式）
        tone = 'formal' if self.reply_settings['formal_tone'] else 'casual'
        
        # 获取模板
        try:
            template_text = self.templates[template_key][tone][language]
        except KeyError:
            # 如果没有找到对应模板，使用通用确认模板
            template_text = self.templates['general_acknowledgment']['formal'][language]
        
        # 准备模板变量
        template_vars = {
            'subject': email_data['subject'],
            'timeframe': '24小时' if language == 'zh' else '24 hours',
            'return_date': (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d'),
            'emergency_contact': 'assistant@company.com'
        }
        
        # 渲染模板
        template = Template(template_text)
        reply_content = template.render(**template_vars)
        
        # 生成完整回复
        reply = {
            'to': email_data['sender'],
            'subject': f"Re: {email_data['subject']}",
            'content': reply_content,
            'template_used': template_key,
            'tone': tone,
            'language': language
        }
        
        return reply

print("✅ 回复生成器定义完成")

## 7. 提醒管理器

In [ ]:
class ReminderManager:
    def __init__(self, config):
        self.config = config
        self.reminder_settings = config['reminder_settings']
        self.reminders = []
    
    def create_reminders(self, email_data, extracted_info):
        """根据提取的信息创建提醒"""
        reminders = []
        
        # 为每个关键日期创建提醒
        for date_str in extracted_info['key_dates']:
            try:
                target_date = datetime.strptime(date_str, '%Y-%m-%d')
                
                # 为每个提前天数创建提醒
                for advance_days in self.reminder_settings['advance_days']:
                    reminder_date = target_date - timedelta(days=advance_days)
                    
                    # 只创建未来的提醒
                    if reminder_date > datetime.now():
                        reminder = {
                            'id': f"{email_data['id']}_{date_str}_{advance_days}",
                            'email_id': email_data['id'],
                            'email_subject': email_data['subject'],
                            'reminder_date': reminder_date,
                            'target_date': target_date,
                            'advance_days': advance_days,
                            'message': f"提醒：{email_data['subject']} - 还有{advance_days}天到期（{date_str}）",
                            'status': 'pending'
                        }
                        reminders.append(reminder)
            except ValueError:
                continue  # 跳过无法解析的日期
        
        # 为待办事项创建提醒
        for todo in extracted_info['todo_items']:
            reminder = {
                'id': f"{email_data['id']}_todo_{hash(todo) % 10000}",
                'email_id': email_data['id'],
                'email_subject': email_data['subject'],
                'reminder_date': datetime.now() + timedelta(hours=2),  # 2小时后提醒
                'target_date': None,
                'advance_days': 0,
                'message': f"待办事项提醒：{todo}",
                'status': 'pending'
            }
            reminders.append(reminder)
        
        self.reminders.extend(reminders)
        return reminders
    
    def get_pending_reminders(self):
        """获取待处理的提醒"""
        now = datetime.now()
        pending = []
        
        for reminder in self.reminders:
            if (reminder['status'] == 'pending' and 
                reminder['reminder_date'] <= now):
                pending.append(reminder)
        
        return pending
    
    def mark_reminder_sent(self, reminder_id):
        """标记提醒已发送"""
        for reminder in self.reminders:
            if reminder['id'] == reminder_id:
                reminder['status'] = 'sent'
                break
    
    def get_reminders_summary(self):
        """获取提醒摘要"""
        total = len(self.reminders)
        pending = len([r for r in self.reminders if r['status'] == 'pending'])
        sent = len([r for r in self.reminders if r['status'] == 'sent'])
        
        return {
            'total': total,
            'pending': pending,
            'sent': sent
        }

print("✅ 提醒管理器定义完成")

## 8. 主程序 - 智能邮件助手

In [ ]:
class EmailSmartAssistant:
    def __init__(self, config, templates):
        self.config = config
        self.templates = templates
        
        # 初始化各个组件
        self.connector = None
        self.classifier = EmailClassifier(config)
        self.extractor = InformationExtractor()
        self.reply_generator = ReplyGenerator(templates, config)
        self.reminder_manager = ReminderManager(config)
        
        # 处理结果存储
        self.processed_emails = []
        self.processing_stats = {
            'total_emails': 0,
            'classified_emails': 0,
            'replies_generated': 0,
            'reminders_created': 0
        }
    
    def connect_email_account(self, account_index=0):
        """连接邮箱账户"""
        if account_index >= len(self.config['email_accounts']):
            console.print("❌ 邮箱账户索引超出范围", style="red")
            return False
        
        account_config = self.config['email_accounts'][account_index]
        self.connector = EmailConnector(account_config)
        
        return self.connector.connect_imap()
    
    def process_emails(self, limit=20):
        """处理邮件的主要流程"""
        if not self.connector:
            console.print("❌ 请先连接邮箱账户", style="red")
            return
        
        console.print("🚀 开始处理邮件...", style="blue")
        
        # 获取邮件
        emails = self.connector.get_emails(limit=limit)
        self.processing_stats['total_emails'] = len(emails)
        
        if not emails:
            console.print("📭 没有找到邮件", style="yellow")
            return
        
        console.print(f"📧 找到 {len(emails)} 封邮件，开始处理...", style="green")
        
        # 处理每封邮件
        for email_data in tqdm(emails, desc="处理邮件"):
            try:
                processed_email = self.process_single_email(email_data)
                self.processed_emails.append(processed_email)
            except Exception as e:
                console.print(f"❌ 处理邮件失败: {str(e)}", style="red")
                continue
        
        console.print("✅ 邮件处理完成！", style="green")
        self.display_processing_summary()
    
    def process_single_email(self, email_data):
        """处理单封邮件"""
        # 1. 邮件分类
        classification = self.classifier.classify_email(email_data)
        self.processing_stats['classified_emails'] += 1
        
        # 2. 信息提取
        extracted_info = self.extractor.generate_summary(email_data)
        
        # 3. 生成回复草稿
        reply_draft = None
        if classification['type'] != 'spam':  # 不为垃圾邮件生成回复
            reply_draft = self.reply_generator.generate_reply(email_data, classification)
            self.processing_stats['replies_generated'] += 1
        
        # 4. 创建提醒
        reminders = []
        if classification['priority'] in ['high', 'medium']:
            reminders = self.reminder_manager.create_reminders(email_data, extracted_info)
            self.processing_stats['reminders_created'] += len(reminders)
        
        # 组装处理结果
        processed_email = {
            'original_email': email_data,
            'classification': classification,
            'extracted_info': extracted_info,
            'reply_draft': reply_draft,
            'reminders': reminders,
            'processed_at': datetime.now().isoformat()
        }
        
        return processed_email
    
    def display_processing_summary(self):
        """显示处理摘要"""
        table = Table(title="📊 邮件处理摘要")
        table.add_column("项目", style="cyan")
        table.add_column("数量", style="magenta")
        
        table.add_row("总邮件数", str(self.processing_stats['total_emails']))
        table.add_row("已分类邮件", str(self.processing_stats['classified_emails']))
        table.add_row("生成回复草稿", str(self.processing_stats['replies_generated']))
        table.add_row("创建提醒", str(self.processing_stats['reminders_created']))
        
        console.print(table)
    
    def get_classification_stats(self):
        """获取分类统计"""
        if not self.processed_emails:
            return {}
        
        stats = {
            'type': {},
            'priority': {},
            'sender_type': {}
        }
        
        for email in self.processed_emails:
            classification = email['classification']
            
            # 统计类型
            email_type = classification['type']
            stats['type'][email_type] = stats['type'].get(email_type, 0) + 1
            
            # 统计优先级
            priority = classification['priority']
            stats['priority'][priority] = stats['priority'].get(priority, 0) + 1
            
            # 统计发件人类型
            sender_type = classification['sender_type']
            stats['sender_type'][sender_type] = stats['sender_type'].get(sender_type, 0) + 1
        
        return stats
    
    def save_results(self, output_dir='output'):
        """保存处理结果"""
        import os
        
        # 创建输出目录
        os.makedirs(f"{output_dir}/reports", exist_ok=True)
        os.makedirs(f"{output_dir}/drafts", exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # 保存处理报告
        report_data = {
            'processing_stats': self.processing_stats,
            'classification_stats': self.get_classification_stats(),
            'reminder_summary': self.reminder_manager.get_reminders_summary(),
            'processed_emails': self.processed_emails,
            'generated_at': datetime.now().isoformat()
        }
        
        with open(f"{output_dir}/reports/email_report_{timestamp}.json", 'w', encoding='utf-8') as f:
            json.dump(report_data, f, ensure_ascii=False, indent=2)
        
        # 保存回复草稿
        drafts = []
        for email in self.processed_emails:
            if email['reply_draft']:
                drafts.append({
                    'original_subject': email['original_email']['subject'],
                    'original_sender': email['original_email']['sender'],
                    'reply': email['reply_draft']
                })
        
        with open(f"{output_dir}/drafts/reply_drafts_{timestamp}.json", 'w', encoding='utf-8') as f:
            json.dump(drafts, f, ensure_ascii=False, indent=2)
        
        console.print(f"✅ 结果已保存到 {output_dir} 目录", style="green")

print("✅ 智能邮件助手主程序定义完成")

## 9. 可视化和报告生成

In [ ]:
def create_visualization(assistant):
    """创建可视化图表"""
    if not assistant.processed_emails:
        console.print("❌ 没有处理过的邮件数据", style="red")
        return
    
    stats = assistant.get_classification_stats()
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('邮件处理分析报告', fontsize=16, fontweight='bold')
    
    # 1. 邮件类型分布
    if stats['type']:
        type_labels = list(stats['type'].keys())
        type_values = list(stats['type'].values())
        
        axes[0, 0].pie(type_values, labels=type_labels, autopct='%1.1f%%', startangle=90)
        axes[0, 0].set_title('邮件类型分布')
    
    # 2. 优先级分布
    if stats['priority']:
        priority_labels = list(stats['priority'].keys())
        priority_values = list(stats['priority'].values())
        
        colors = {'high': 'red', 'medium': 'orange', 'low': 'green'}
        bar_colors = [colors.get(label, 'blue') for label in priority_labels]
        
        axes[0, 1].bar(priority_labels, priority_values, color=bar_colors)
        axes[0, 1].set_title('邮件优先级分布')
        axes[0, 1].set_ylabel('数量')
    
    # 3. 发件人类型分布
    if stats['sender_type']:
        sender_labels = list(stats['sender_type'].keys())
        sender_values = list(stats['sender_type'].values())
        
        axes[1, 0].bar(sender_labels, sender_values)
        axes[1, 0].set_title('发件人类型分布')
        axes[1, 0].set_ylabel('数量')
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 4. 处理统计
    process_labels = ['总邮件', '已分类', '生成回复', '创建提醒']
    process_values = [
        assistant.processing_stats['total_emails'],
        assistant.processing_stats['classified_emails'],
        assistant.processing_stats['replies_generated'],
        assistant.processing_stats['reminders_created']
    ]
    
    axes[1, 1].bar(process_labels, process_values, color='skyblue')
    axes[1, 1].set_title('处理统计')
    axes[1, 1].set_ylabel('数量')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

def display_sample_results(assistant, num_samples=3):
    """显示处理结果样例"""
    if not assistant.processed_emails:
        console.print("❌ 没有处理过的邮件数据", style="red")
        return
    
    console.print("\n📋 处理结果样例:", style="bold blue")
    
    for i, email in enumerate(assistant.processed_emails[:num_samples]):
        console.print(f"\n--- 邮件 {i+1} ---", style="yellow")
        
        # 原始邮件信息
        original = email['original_email']
        console.print(f"主题: {original['subject']}", style="cyan")
        console.print(f"发件人: {original['sender']}", style="cyan")
        
        # 分类结果
        classification = email['classification']
        console.print(f"类型: {classification['type']} | 优先级: {classification['priority']} | 发件人类型: {classification['sender_type']}", style="green")
        
        # 提取的信息
        extracted = email['extracted_info']
        if extracted['key_dates']:
            console.print(f"关键日期: {', '.join(extracted['key_dates'])}", style="magenta")
        if extracted['todo_items']:
            console.print(f"待办事项: {extracted['todo_items'][0][:50]}...", style="magenta")
        
        # 回复草稿
        if email['reply_draft']:
            reply = email['reply_draft']
            console.print(f"回复草稿 ({reply['tone']}, {reply['language']}): {reply['content'][:100]}...", style="white")
        
        # 提醒
        if email['reminders']:
            console.print(f"创建了 {len(email['reminders'])} 个提醒", style="yellow")

print("✅ 可视化和报告功能定义完成")

## 10. 演示和测试

In [ ]:
# 创建演示数据（如果无法连接真实邮箱）
def create_demo_data():
    """创建演示数据"""
    demo_emails = [
        {
            'id': '1',
            'subject': '紧急：项目进度汇报会议安排',
            'sender': 'manager@company.com',
            'date': '2024-01-15 09:00:00',
            'body': '各位同事，请准备明天下午2点的项目进度汇报会议。需要准备本周工作总结和下周计划。截止时间：2024-01-16 14:00。请确认参会。'
        },
        {
            'id': '2',
            'subject': '客户咨询：产品功能详情',
            'sender': 'customer@client.com',
            'date': '2024-01-15 10:30:00',
            'body': '您好，我对贵公司的产品很感兴趣，希望了解更多功能详情。请问可以安排一次产品演示吗？我的联系方式：13800138000。期待您的回复。'
        },
        {
            'id': '3',
            'subject': '系统维护通知',
            'sender': 'noreply@system.com',
            'date': '2024-01-15 11:00:00',
            'body': '系统将于2024-01-20 02:00-04:00进行维护升级，期间服务可能中断。请提前做好准备工作。如有疑问请联系技术支持。'
        },
        {
            'id': '4',
            'subject': '限时优惠！立即购买享受8折优惠',
            'sender': 'promotion@ads.com',
            'date': '2024-01-15 12:00:00',
            'body': '亲爱的用户，我们的产品正在进行限时促销活动！现在购买可享受8折优惠，机会难得，不要错过！点击链接立即购买。'
        },
        {
            'id': '5',
            'subject': '个人：周末聚餐安排',
            'sender': 'friend@personal.com',
            'date': '2024-01-15 13:00:00',
            'body': '嗨！这个周末我们一起聚餐吧，时间定在周六晚上7点，地点在市中心的那家川菜馆。请确认是否能参加，我好提前订位。'
        }
    ]
    
    return demo_emails

def run_demo():
    """运行演示程序"""
    console.print("🎯 开始演示智能邮件助手", style="bold blue")
    
    # 检查配置
    if not config or not templates:
        console.print("❌ 配置或模板加载失败，无法运行演示", style="red")
        return
    
    # 创建助手实例
    assistant = EmailSmartAssistant(config, templates)
    
    # 使用演示数据
    console.print("📧 使用演示数据进行测试...", style="yellow")
    demo_emails = create_demo_data()
    
    # 处理演示邮件
    assistant.processing_stats['total_emails'] = len(demo_emails)
    
    for email_data in tqdm(demo_emails, desc="处理演示邮件"):
        try:
            processed_email = assistant.process_single_email(email_data)
            assistant.processed_emails.append(processed_email)
        except Exception as e:
            console.print(f"❌ 处理邮件失败: {str(e)}", style="red")
            continue
    
    # 显示结果
    console.print("\n✅ 演示处理完成！", style="green")
    assistant.display_processing_summary()
    
    # 显示样例结果
    display_sample_results(assistant)
    
    # 创建可视化
    create_visualization(assistant)
    
    # 保存结果
    assistant.save_results()
    
    return assistant

print("✅ 演示程序准备完成")

## 11. 运行智能邮件助手

In [ ]:
# 运行演示
assistant = run_demo()

## 12. 实际邮箱连接（可选）

In [ ]:
# 如果要连接真实邮箱，请先配置 config/email_config.json 文件
# 然后取消注释下面的代码

# def run_with_real_email():
#     """使用真实邮箱运行"""
#     console.print("🔗 连接真实邮箱...", style="blue")
#     
#     # 创建助手实例
#     assistant = EmailSmartAssistant(config, templates)
#     
#     # 连接邮箱
#     if not assistant.connect_email_account(0):  # 使用第一个邮箱账户
#         console.print("❌ 邮箱连接失败", style="red")
#         return None
#     
#     # 处理邮件
#     assistant.process_emails(limit=10)  # 处理最新10封邮件
#     
#     # 显示结果
#     display_sample_results(assistant)
#     create_visualization(assistant)
#     assistant.save_results()
#     
#     # 关闭连接
#     assistant.connector.close_connections()
#     
#     return assistant

# # 运行真实邮箱处理
# real_assistant = run_with_real_email()

console.print("\n🎉 智能邮件助手演示完成！", style="bold green")
console.print("\n📝 使用说明:", style="bold yellow")
console.print("1. 修改 config/email_config.json 配置你的邮箱信息")
console.print("2. 取消注释上面的真实邮箱连接代码")
console.print("3. 运行代码开始处理你的邮件")
console.print("4. 查看 output 目录中的处理报告和回复草稿")